# Solution: Feature Scaling -- Breast Cancer Dataset

**Exercises from notebook `02_feature_scaling.ipynb`**

Jingxu Li - 320230942071


## Exercise 1: KNN with and without StandardScaler

Load sklearn.datasets.load_breast_cancer. Fit a KNeighborsClassifier with and without StandardScaler. Compare 5-fold cross-validated accuracy.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

data = load_breast_cancer()
X, y = data.data, data.target

knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_scaled = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

raw_scores = cross_val_score(knn_raw, X, y, cv=5, scoring='accuracy')
scaled_scores = cross_val_score(knn_scaled, X, y, cv=5, scoring='accuracy')

print(f"KNN without scaling -- Accuracy: {raw_scores.mean():.4f} (+/- {raw_scores.std() * 2:.4f})")
print(f"KNN with StandardScaler -- Accuracy: {scaled_scores.mean():.4f} (+/- {scaled_scores.std() * 2:.4f})")
print(f"Improvement: {(scaled_scores.mean() - raw_scores.mean()) * 100:.2f} percentage points")


KNN without scaling -- Accuracy: 0.9279 (+/- 0.0435)
KNN with StandardScaler -- Accuracy: 0.9649 (+/- 0.0192)
Improvement: 3.69 percentage points


## Exercise 2: Outliers -- StandardScaler vs RobustScaler

Introduce 5 extreme outliers into the mean radius feature. Compare how StandardScaler vs RobustScaler handle them -- measure the impact on downstream KNN accuracy.


In [8]:
from sklearn.preprocessing import RobustScaler

X_out = X.copy()
np.random.seed(42)
outlier_idx = np.random.choice(len(X_out), 5, replace=False)
X_out[outlier_idx, 0] = X_out[outlier_idx, 0] * 10000  

knn_standard = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

knn_robust = Pipeline([
    ('scaler', RobustScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

standard_scores = cross_val_score(knn_standard, X_out, y, cv=5, scoring='accuracy')
robust_scores = cross_val_score(knn_robust, X_out, y, cv=5, scoring='accuracy')

print(f"StandardScaler (with outliers) -- Accuracy: {standard_scores.mean():.4f} (+/- {standard_scores.std() * 2:.4f})")
print(f"RobustScaler (with outliers) -- Accuracy: {robust_scores.mean():.4f} (+/- {robust_scores.std() * 2:.4f})")
print(f"\nAfter injecting 5 extreme outliers (×10000), StandardScaler achieved {standard_scores.mean():.4f}")
print(f"RobustScaler achieved {robust_scores.mean():.4f}. ")


StandardScaler (with outliers) -- Accuracy: 0.9649 (+/- 0.0192)
RobustScaler (with outliers) -- Accuracy: 0.9578 (+/- 0.0341)

After injecting 5 extreme outliers (×10000), StandardScaler achieved 0.9649
RobustScaler achieved 0.9578. 


The difference is small because only 5 outliers in 569 samples have limited impact. RobustScaler's advantage becomes
more apparent with more outliers or more extreme values, as it uses median/IQR
instead of mean/std, making it resistant to extreme values that distort StandardScaler.

## Exercise 3: Custom Percentile Scaler

Implement a custom scaler class that scales to [-1, 1] using the 5th and 95th percentiles instead of min and max. Show it handles outliers better than MinMaxScaler.


In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

class PercentileScaler(BaseEstimator, TransformerMixin):
    def __init__(self, low=5, high=95):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        self.p5_ = np.percentile(X, self.low, axis=0)
        self.p95_ = np.percentile(X, self.high, axis=0)
        self.range_ = self.p95_ - self.p5_
        self.range_[self.range_ == 0] = 1
        return self

    def transform(self, X, y=None):
        scaled = -1 + 2 * (X - self.p5_) / self.range_
        return np.clip(scaled, -1, 1)

X_outlier_col = X_out[:, [0]]

ps = PercentileScaler()
X_ps = ps.fit_transform(X_outlier_col)

from sklearn.preprocessing import MinMaxScaler
mms = MinMaxScaler(feature_range=(-1, 1))
X_mms = mms.fit_transform(X_outlier_col)

print("MinMaxScaler -- scaled outlier col (first 7 rows):")
print(np.round(X_mms[:7].ravel(), 4))
print(f"  Outlier row {outlier_idx[0]} value: {X_mms[outlier_idx[0]][0]:.4f}")
print(f"  Most values compressed to: [{X_mms[:,0].min():.4f}, {X_mms[:,0].max():.4f}]")

# MinMaxScaler KNN for comparison
knn_mms = Pipeline([
    ('scaler', MinMaxScaler(feature_range=(-1, 1))),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
mms_scores = cross_val_score(knn_mms, X_out, y, cv=5, scoring='accuracy')
print(f"\nMinMaxScaler KNN -- Accuracy: {mms_scores.mean():.4f} (+/- {mms_scores.std() * 2:.4f})")

print(f"\nPercentileScaler (5th-95th) -- scaled outlier col (first 7 rows):")
print(np.round(X_ps[:7].ravel(), 4))
print(f"  Outlier row {outlier_idx[0]} value: {X_ps[outlier_idx[0]][0]:.4f}")
print(f"  Most values in range: [{X_ps[:,0].min():.4f}, {X_ps[:,0].max():.4f}]")

# Compare on KNN accuracy
knn_ps = Pipeline([
    ('scaler', PercentileScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
ps_scores = cross_val_score(knn_ps, X_out, y, cv=5, scoring='accuracy')
print(f"\nPercentileScaler KNN -- Accuracy: {ps_scores.mean():.4f} (+/- {ps_scores.std() * 2:.4f})")


MinMaxScaler -- scaled outlier col (first 7 rows):
[-0.9999 -0.9999 -0.9999 -1.     -0.9999 -0.9999 -0.9999]
  Outlier row 204 value: 0.3168
  Most values compressed to: [-1.0000, 1.0000]

MinMaxScaler KNN -- Accuracy: 0.9719 (+/- 0.0172)

PercentileScaler (5th-95th) -- scaled outlier col (first 7 rows):
[ 0.4955  0.9516  0.796  -0.6658  0.9021 -0.4837  0.5415]
  Outlier row 204 value: 1.0000
  Most values in range: [-1.0000, 1.0000]

PercentileScaler KNN -- Accuracy: 0.9666 (+/- 0.0302)


## Exercise 4: Pipeline Leakage Verification

Write a test that verifies that a Pipeline(StandardScaler, Ridge) fitted on training data uses only training statistics -- check pipe.named_steps['scale'].mean_ vs X_train.mean().


In [10]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

pipe.fit(X_train, y_train)

scaler_mean = pipe.named_steps['scale'].mean_
train_mean = X_train.mean(axis=0)
test_mean = X_test.mean(axis=0)

print("Feature | Scaler Mean | X_train Mean | Match?")
print("-" * 55)
for i in range(min(5, X.shape[1])):
    match = "YES" if np.allclose(scaler_mean[i], train_mean[i]) else "NO"
    print(f"   {i:2d}    |   {scaler_mean[i]:8.4f}  |   {train_mean[i]:8.4f}    |   {match}")

all_match = np.allclose(scaler_mean, train_mean)
print(f"\nAll features match: {all_match}")

test_matches = np.allclose(scaler_mean, test_mean)
print(f"Scaler matches X_test mean (would indicate leakage): {test_matches}")
print("[OK] Pipeline scaler uses ONLY training statistics -- no data leakage.")


Feature | Scaler Mean | X_train Mean | Match?
-------------------------------------------------------
    0    |    14.1176  |    14.1176    |   YES
    1    |    19.1850  |    19.1850    |   YES
    2    |    91.8822  |    91.8822    |   YES
    3    |   654.3776  |   654.3776    |   YES
    4    |     0.0957  |     0.0957    |   YES

All features match: True
Scaler matches X_test mean (would indicate leakage): False
[OK] Pipeline scaler uses ONLY training statistics -- no data leakage.
